# Question 6 — Plot GameStop Stock Graph

Download GameStop stock-price history, scrape annual revenue, define `make_graph()`, and plot both series with a descriptive title. The upper panel shows daily closing price; the lower panel shows annual revenue in USD millions.

In [ ]:
%pip install -q yfinance pandas requests beautifulsoup4 lxml plotly

In [ ]:
import re
from io import StringIO

import pandas as pd
import requests
import yfinance as yf
from bs4 import BeautifulSoup
from IPython.display import display


## Download GameStop stock data

In [ ]:
# Download all available daily price history for GME from Yahoo Finance.
gme_data = yf.Ticker("GME").history(period="max", auto_adjust=False)
if gme_data.empty:
    raise RuntimeError("No stock data was returned. Check your internet connection and ticker symbol.")
gme_data.reset_index(inplace=True)
gme_data.to_csv("gme_stock_data.csv", index=False)
print("Rows:", len(gme_data))
display(gme_data.head())

## Define the revenue web scraper

In [ ]:
def scrape_annual_revenue(ticker: str) -> pd.DataFrame:
    """Scrape annual revenue from Stock Analysis. Revenue is returned in USD millions."""
    symbol = ticker.strip().lower()
    url = f"https://stockanalysis.com/stocks/{symbol}/revenue/"
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/120.0.0.0 Safari/537.36"
        )
    }
    response = requests.get(url, headers=headers, timeout=30)
    response.raise_for_status()

    try:
        tables = pd.read_html(StringIO(response.text))
    except ValueError as exc:
        raise RuntimeError(
            f"No readable HTML table was found on {url}. The page may have changed "
            "its layout or may be rate-limiting automated requests."
        ) from exc

    revenue_table = None
    for table in tables:
        if isinstance(table.columns, pd.MultiIndex):
            table.columns = [
                " ".join(str(x) for x in col if str(x) != "nan").strip()
                for col in table.columns
            ]
        columns = list(table.columns)
        date_col = next(
            (c for c in columns if "fiscal year" in str(c).lower()
             or "period end" in str(c).lower()
             or str(c).strip().lower() in {"date", "year"}),
            None,
        )
        revenue_col = next(
            (c for c in columns if "revenue" in str(c).lower()), None
        )
        if date_col is not None and revenue_col is not None:
            revenue_table = table[[date_col, revenue_col]].copy()
            break

    if revenue_table is None:
        raise RuntimeError(
            f"Could not find Date/Year and Revenue columns on {url}. "
            "Inspect pd.read_html(StringIO(response.text)) and update the scraper."
        )

    revenue_table.columns = ["Date", "Revenue"]
    revenue_table["Date"] = pd.to_datetime(revenue_table["Date"], errors="coerce")

    def to_usd_millions(value):
        if pd.isna(value):
            return float("nan")
        text = str(value).strip().replace("$", "").replace(",", "")
        if text in {"", "-", "—", "nan", "None"}:
            return float("nan")
        suffix = text[-1].upper() if text[-1].isalpha() else ""
        number_text = re.sub(r"[^0-9.\-]", "", text)
        if not number_text:
            return float("nan")
        number = float(number_text)
        if suffix == "B":
            number *= 1000  # billions to millions
        elif suffix == "K":
            number /= 1000  # thousands to millions
        return number  # M or plain values are already in millions

    revenue_table["Revenue"] = revenue_table["Revenue"].apply(to_usd_millions)
    revenue_table = revenue_table.dropna(subset=["Date", "Revenue"])
    revenue_table = revenue_table.sort_values("Date").reset_index(drop=True)
    if revenue_table.empty:
        raise RuntimeError(f"No usable annual revenue rows were found for {ticker.upper()}.")
    return revenue_table


## Scrape GameStop annual revenue

In [ ]:
gme_revenue = scrape_annual_revenue("GME")
display(gme_revenue.tail())
gme_revenue.to_csv("gme_revenue.csv", index=False)

## Define the make_graph function

In [ ]:
def make_graph(stock_data: pd.DataFrame, revenue_data: pd.DataFrame, title: str):
    """Plot daily closing share price and annual revenue in two stacked panels."""
    from plotly.subplots import make_subplots
    import plotly.graph_objects as go

    stock = stock_data.copy()
    revenue = revenue_data.copy()
    if "Date" not in stock.columns:
        stock = stock.reset_index()
    if "Date" not in revenue.columns:
        revenue = revenue.reset_index()

    stock_date_col = "Date" if "Date" in stock.columns else stock.columns[0]
    revenue_date_col = "Date" if "Date" in revenue.columns else revenue.columns[0]
    stock[stock_date_col] = pd.to_datetime(
        stock[stock_date_col], errors="coerce", utc=True
    ).dt.tz_convert(None)
    revenue[revenue_date_col] = pd.to_datetime(
        revenue[revenue_date_col], errors="coerce"
    )
    stock["Close"] = pd.to_numeric(stock["Close"], errors="coerce")
    revenue["Revenue"] = pd.to_numeric(revenue["Revenue"], errors="coerce")
    stock = stock.dropna(subset=[stock_date_col, "Close"]).sort_values(stock_date_col)
    revenue = revenue.dropna(subset=[revenue_date_col, "Revenue"]).sort_values(revenue_date_col)

    fig = make_subplots(
        rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.12,
        subplot_titles=("Historical Share Price", "Annual Revenue")
    )
    fig.add_trace(
        go.Scatter(
            x=stock[stock_date_col], y=stock["Close"],
            mode="lines", name="Closing share price"
        ), row=1, col=1
    )
    fig.add_trace(
        go.Bar(
            x=revenue[revenue_date_col], y=revenue["Revenue"],
            name="Annual revenue"
        ), row=2, col=1
    )
    fig.update_yaxes(title_text="Share Price (USD)", row=1, col=1)
    fig.update_yaxes(title_text="Revenue (USD millions)", row=2, col=1)
    fig.update_xaxes(title_text="Date", row=2, col=1)
    fig.update_layout(
        title=title, height=850, showlegend=False,
        hovermode="x unified", template="plotly_white"
    )
    fig.show()
    return fig


## Plot GameStop stock data with a title

In [ ]:
gme_graph = make_graph(
    gme_data,
    gme_revenue,
    "GameStop Stock Price and Revenue"
)

## Submission screenshot

Capture the executed plotting cell and the complete two-panel Plotly chart. Use the camera icon in Plotly's toolbar to export a PNG if an image file is required.